In [1]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from z3 import *
from typing import List
import pickle 

In [2]:
loaded_model = joblib.load('logistic_model.pkl')


In [3]:
weights= loaded_model.coef_
biases= loaded_model.intercept_

In [4]:
weights.shape

(1, 10)

In [5]:
def build_model(weights, biases):
    model=weights[0].tolist()
    return model, biases[0]

In [6]:
Model,bias= build_model(weights,biases)

In [7]:
Model

[0.5523376887975173,
 0.12473125176881826,
 -0.06672991864847844,
 0.6028524002339479,
 -0.4465428057350922,
 1.356439883895435,
 0.11945026083144535,
 -0.7911976147781454,
 -0.2761451710853004,
 0.1576799952621751]

In [8]:
bias

0.4240261394219779

In [9]:
def create_smt_file(name_file):
    file= open(name_file,'w')
    return file
def close_smt_file(f):
    f.close()

In [10]:
file= create_smt_file('model.smt2')

In [11]:
def initialize_variables(model):
    file.write("\n")
    
    N,D= model.shape
    for i in range(D):
        file.write("(declare-fun i"+str(0)+str(i)+"() Real)\n")
        file.write("\n")
        
    for j in range(N):
        file.write("(declare-fun y"+str(0)+str(j)+"() Real)\n")
        file.write("\n") 
        
    for k in range(N):
        file.write("(declare-fun o"+str(0)+str(k)+"() Real)\n")
        file.write("\n") 
    file.write("(declare-fun decision() Int)\n")
initialize_variables(weights)

In [12]:
def constraint_input_output(model):
    N,D= model.shape
    file.write("\n")
    for i in range(D):
        file.write("(assert (or \n")
        file.write("(= i0"+str(i)+" 1) (= i0"+str(i)+" 0)\n")
        file.write("))\n")
                
    file.write("(assert (or \n")
    file.write("(= decision 1) (= decision 0)\n")
    file.write("))\n")

constraint_input_output(weights)

In [13]:
def weight_computation(model,list_coeficients,bias):
    N,D= model.shape
    file.write("\n") 
    
    for i in range(N):
        file.write("(assert (= \n")
        file.write("y"+str(0)+str(i)+" \n")
        file.write("(+\n")
        
        k=0
        for j in list_coeficients:
            file.write("(* " +str(j)+" i"+str(0)+str(k)+")\n")
            k+=1
        file.write("(+ " +str(bias)+" 0)\n")
        file.write(")))\n")

weight_computation(weights,Model,bias)

In [14]:
def compute_output(model):
    N,D= model.shape
    file.write("\n")
    for i in range(N):
        file.write("(assert (and \n")
        file.write("(or (not \n")
        file.write("(> y"+str(0)+str(i)+ " 0))\n")
        file.write("(= o"+str(0)+str(i)+ " y"+str(0)+str(i)+"))\n")
        file.write("(or (not \n")
        file.write("(<= y"+str(0)+str(i)+ " 0))\n")
        file.write("(= o"+str(0)+str(i)+ " 0))\n")
        file.write("))\n")       
            
compute_output(weights)

In [15]:
def compute_decision(threshold):
    file.write("\n") 
    file.write("(assert (and \n")
    file.write("(or (not \n")
    file.write("(> o"+str(0)+"0 "+str(threshold)+"))\n")
    file.write("(= decision 1))\n")
    file.write("(or (not \n")
    file.write("(<= o"+str(0)+"0 "+str(threshold)+"))\n")
    file.write("(= decision 0))\n")
    file.write("))\n")
    
compute_decision(0)

In [16]:
close_smt_file(file)

#### Create test profiles
This notebook allows to compute the decisions of the test set with the trained neural network.

In [17]:
def load_profiles(path):
    data= pd.read_csv(path, sep=',')
    data= data.drop(['Unnamed: 0','Risk'], axis=1)
    return(data)
profile= load_profiles('binarized_GCD')
profs= profile.values
profs[0:].shape

(1000, 10)

In [18]:
def compute_decisions(model,profiles):
    individuals=[]
    predictions= model.predict(profiles[0:])
    return predictions
predictions=compute_decisions(loaded_model,profs)

In [19]:
def save_individuals(dataframe, predictions,new_column):
    dataframe[new_column] = predictions
    dataframe.to_csv("GCD_individuals", sep=',')
    return dataframe
df= save_individuals(profile,predictions,'y')

In [20]:
path= "model.smt2"
def load_solver(path):
    s= Solver()
    s.from_file(path)
    return s
solver= load_solver(path)

In [21]:
def check_sat(s, profil):
    predictions=[]
    for prof in profil:
        s.push()
        for i in range(len(prof)):
            s.add(Real('i0'+str(i)) ==  prof[i])
        s.add(Int('decision') == 0)
        resp= s.check()
        if resp== unsat:
            predictions.append(1)
        else:
            predictions.append(0)
        s.pop()
    return predictions
predictions_smt=check_sat(solver,profs)
data=save_individuals(df,predictions_smt,'y_smt')

In [22]:
data

,A,G,J,H,S,B,C,D,P,M,y,y_smt
0,1,1,1,1,0,0,0,0,0,1,1,1
1,0,0,1,1,1,0,1,1,0,0,0,0
2,1,1,0,1,1,1,0,0,0,1,1,1
3,1,1,1,0,1,0,1,1,0,1,1,1
4,1,1,1,0,1,0,1,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,1,1,1,0,0,0,0,1,1
996,1,1,0,1,1,0,1,1,1,0,1,1
997,1,1,1,1,1,1,0,0,0,1,1,1
998,0,1,1,0,1,0,0,1,0,1,0,0


In [23]:
from sklearn.metrics import accuracy_score
acc = accuracy_score(data['y'], data['y_smt'])
acc

1.0